---
## 1. Setup

In [ ]:
!pip install transformers accelerate -q

import numpy as np
import pandas as pd
import re
import os
import gc
import warnings
import unicodedata
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

import psutil

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = False
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    cap = torch.cuda.get_device_capability()
    print(f'Compute Capability: sm_{cap[0]}{cap[1]}')

def ram_usage():
    p = psutil.Process(os.getpid())
    print(f'  RAM: {p.memory_info().rss/1e9:.2f} GB')
    if DEVICE.type == 'cuda':
        print(f'  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

---
## 2. Load Data

In [ ]:
DATA_DIR = '/kaggle/input/datasets/namnguynnnn/sentence-classification/Dataset/'
train = pd.read_csv(DATA_DIR + 'train.csv')
test  = pd.read_csv(DATA_DIR + 'test.csv')

train['Text'] = train['Text'].fillna('')
test['Text']  = test['Text'].fillna('')

n_before = len(train)
train = train.drop_duplicates(subset=['Text']).reset_index(drop=True)
print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Removed {n_before - len(train)} duplicates')
display(train.head(3))

---
## 3. EDA

In [ ]:
# Label distribution
label_counts = train['Label'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x='Label', data=train, palette='Set2', ax=axes[0])
axes[0].set_title('Label Distribution')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom')
label_counts.plot.pie(autopct='%1.1f%%', ax=axes[1], labels=['Positive','Negative'],
                      colors=['#66c2a5','#fc8d62'], startangle=90)
axes[1].set_ylabel('')
plt.tight_layout(); plt.show()

ratio = label_counts.max() / label_counts.min()
print(f'Imbalance ratio: {ratio:.2f}')
print('✅ Balanced' if ratio < 1.5 else '⚠️  Imbalanced → dùng class_weight="balanced"')

In [ ]:
# Word count distribution
word_count = train['Text'].map(lambda x: len(x.split()))
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(x=word_count, hue=train['Label'], bins=50, ax=ax)
ax.set_title('Word Count Distribution by Label')
plt.tight_layout(); plt.show()

for pct in [50, 90, 95, 99]:
    print(f'{pct:3}th percentile: {np.percentile(word_count, pct):.0f} words')

RECOMMENDED_MAX_LEN = min(int(np.percentile(word_count, 95)) * 2, 256)
print(f'\n→ Recommended MAX_LEN = {RECOMMENDED_MAX_LEN}')
del word_count; gc.collect()

---
## 4. Text Preprocessing

### Điểm mới so với notebook trước:
1. **Unicode normalization (NFKC)** — xử lý ký tự đặc biệt, full-width chars
2. **Number replacement `<NUM>`** — giảm noise, giúp model tổng quát hóa
3. **Negation handling `NOT_`** — kỹ thuật quan trọng nhất từ Lab 3.1
   - `"I do not like"` → `"I do not NOT_like"` → TF-IDF bắt được `NOT_like` là feature
   - `"doesn't good"` → expand → `"does not NOT_good"`
   - Tăng đáng kể F1 với review có phủ định

In [ ]:
STOPWORDS  = set(stopwords.words('english'))
KEEP_WORDS = {'not', 'no', 'never', 'neither', 'nor', 'hardly', 'barely',
              "n't", 'nobody', 'nothing'}  # ← Thêm từ Lab 3.1
STOPWORDS -= KEEP_WORDS

CONTRACTIONS = {
    "don't":"do not","doesn't":"does not","didn't":"did not",
    "won't":"will not","wouldn't":"would not","can't":"cannot",
    "couldn't":"could not","shouldn't":"should not",
    "isn't":"is not","aren't":"are not","wasn't":"was not",
    "weren't":"were not","hadn't":"had not","hasn't":"has not",
    "haven't":"have not","mustn't":"must not","needn't":"need not",
    "it's":"it is","he's":"he is","she's":"she is",
    "i've":"i have","i'm":"i am","i'll":"i will","i'd":"i would",
    "that's":"that is","there's":"there is","they're":"they are",
    "we're":"we are","you're":"you are","they've":"they have",
}

# Lab 3.1: Negation words và window size
NEGATION_WORDS = {'not', "n't", 'no', 'never', 'neither', 'nobody', 'nothing'}
NEGATION_WINDOW = 3  # Số token sau negation được prepend NOT_


def normalize_text(text: str) -> str:
    """Lab 3.1: Unicode normalization."""
    # NFKC: chuẩn hóa full-width chars (ｈｅｌｌｏ → hello)
    text = unicodedata.normalize('NFKC', text)
    text = ' '.join(text.split())  # chuẩn hóa whitespace
    return text


def handle_negation(tokens: list, window_size: int = 3) -> list:
    """
    Lab 3.1: Prepend NOT_ to tokens after negation words.
    
    Example:
        ["I", "do", "not", "like", "this", "product"]
        → ["I", "do", "not", "NOT_like", "NOT_this", "NOT_product"]
    """
    result = []
    negate_count = 0

    for token in tokens:
        is_negation = (token.lower() in NEGATION_WORDS or
                       token.lower().endswith("n't"))
        if is_negation:
            result.append(token)
            negate_count = window_size
        elif negate_count > 0:
            result.append(f'NOT_{token}')  # ← Key technique từ Lab 3.1
            negate_count -= 1
        else:
            result.append(token)

    return result


def clean_text_for_tfidf(text: str, use_negation: bool = True) -> str:
    """
    Pipeline hoàn chỉnh cho TF-IDF.
    KHÔNG dùng cho BERT (BERT dùng text gốc).
    
    Thứ tự xử lý:
    1. Unicode normalization (Lab 3.1)
    2. Lowercase
    3. Bỏ URL, HTML
    4. Expand contractions
    5. Replace numbers với <NUM> (Lab 3.1)
    6. Tokenize + Negation handling (Lab 3.1)
    7. Reconstruct string
    """
    if not isinstance(text, str): return ''

    # 1. Unicode normalization
    text = normalize_text(text)

    # 2. Lowercase
    text = text.lower()

    # 3. Bỏ URL, HTML
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)

    # 4. Expand contractions (trước khi thay số để giữ n't)
    for c, f in CONTRACTIONS.items():
        text = text.replace(c, f)

    # 5. Replace numbers — Lab 3.1 insight
    # "spent $200" → "spent $<NUM>"  không xóa context xung quanh số
    text = re.sub(r'\d+', '<NUM>', text)

    # 6. Tokenize + Negation handling
    if use_negation:
        tokens = word_tokenize(text)
        tokens = handle_negation(tokens, window_size=NEGATION_WINDOW)
        text = ' '.join(tokens)

    # 7. Bỏ ký tự đặc biệt (giữ lại <NUM> bằng cách xử lý sau)
    text = re.sub(r'[^a-z0-9<>_\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Test
test_cases = [
    "I do not like this product at all!",
    "This isn't good enough for the price.",
    "I spent $200 and it's totally worth it!",
    "AMAZING service! The staff was so helpful.",
    "Terrible. Never buying again.",
]
print('=== Negation Handling Demo ===')
for t in test_cases:
    print(f'BEFORE: {t}')
    print(f'AFTER:  {clean_text_for_tfidf(t)}')
    print()

In [ ]:
# Apply preprocessing
print('Preprocessing...')
train['clean_text'] = train['Text'].apply(clean_text_for_tfidf)
test['clean_text']  = test['Text'].apply(clean_text_for_tfidf)

print(f'Done! Sample:')
print(f'ORIG:  {train["Text"].iloc[0][:80]}')
print(f'CLEAN: {train["clean_text"].iloc[0][:80]}')
ram_usage()

---
## 5. Word Shape Meta-Features
dùng word shape cho NER (ví dụ `Hello → Xx`, `HELLO → X`).
Ấp dụng cho sentiment: đếm số lượng CAPS words, exclamation marks như signal.

In [ ]:
def get_word_shape_short(word: str) -> str:
    """Lab 3.2: Condensed word shape."""
    shape = []
    for char in word:
        if char.isupper():   shape.append('X')
        elif char.islower(): shape.append('x')
        elif char.isdigit(): shape.append('d')
        else:                shape.append(char)
    # Collapse consecutive same chars
    condensed = [shape[0]] if shape else []
    for c in shape[1:]:
        if c != condensed[-1]:
            condensed.append(c)
    return ''.join(condensed)


def extract_meta_features(df: pd.DataFrame) -> np.ndarray:
    """
    Meta-features kết hợp từ Lab 3.1 (negation, punctuation) và Lab 3.2 (word shape).
    """
    feats = pd.DataFrame()

    # --- Độ dài ---
    feats['word_count']    = df['Text'].map(lambda x: len(x.split()))
    feats['char_count']    = df['Text'].map(len)
    feats['avg_word_len']  = df['Text'].map(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )

    # --- Lab 3.1: Punctuation signals ---
    feats['exclaim_count'] = df['Text'].str.count(r'!')
    feats['question_count']= df['Text'].str.count(r'\?')
    feats['ellipsis_count']= df['Text'].str.count(r'\.\.\.')

    # --- Lab 3.1: Negation count ---
    neg_pattern = r'\b(not|no|never|neither|hardly|barely|n\'t)\b'
    feats['negation_count']= df['Text'].str.lower().str.count(neg_pattern)

    # --- Lab 3.2: Word shape features ---
    # CAPS ratio: tỷ lệ từ viết hoa (AMAZING!) → signal mạnh
    feats['caps_ratio'] = df['Text'].map(
        lambda x: sum(1 for w in x.split() if w.isupper() and len(w) > 1) / max(len(x.split()), 1)
    )
    # Title case ratio: "Great Product" → thường positive
    feats['title_ratio'] = df['Text'].map(
        lambda x: sum(1 for w in x.split() if w.istitle()) / max(len(x.split()), 1)
    )

    # --- Domain-specific (từ data bác sĩ) ---
    feats['dr_mentions']  = df['Text'].str.lower().str.count(r'\bdr\.?\b')
    feats['year_mentions']= df['Text'].str.count(r'\d+\s*years?')
    feats['star_mentions']= df['Text'].str.count(r'\b[1-5]\s*stars?\b')

    return feats.fillna(0).values


# Test
print('Meta-feature shapes:')
meta_train = extract_meta_features(train)
meta_test  = extract_meta_features(test)
print(f'Train: {meta_train.shape} | Test: {meta_test.shape}')
print('\nSample meta-features (first row):')
print(meta_train[0])

---
## 6. TF-IDF + ML — Với Negation Handling

> **Tại sao negation handling giúp TF-IDF?**
>
> Không có negation: `"not good"` → feature `not` (negative) + `good` (positive) → confused
>
> Có negation: `"not good"` → feature `not` + `NOT_good` → model học được `NOT_good` là negative

In [ ]:
print('Creating TF-IDF features...')
ram_usage()

vec_word = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30_000,
    sublinear_tf=True,
    min_df=2,
    analyzer='word',
    dtype=np.float32
)
vec_char = TfidfVectorizer(
    ngram_range=(3, 4),
    max_features=20_000,
    sublinear_tf=True,
    min_df=2,
    analyzer='char_wb',
    dtype=np.float32
)

X_train_w = vec_word.fit_transform(train['clean_text'])
X_test_w  = vec_word.transform(test['clean_text'])
X_train_c = vec_char.fit_transform(train['clean_text'])
X_test_c  = vec_char.transform(test['clean_text'])

# TF-IDF
X_train_tfidf = sp.hstack([X_train_w, X_train_c], format='csr')
X_test_tfidf  = sp.hstack([X_test_w,  X_test_c],  format='csr')
del X_train_w, X_train_c, X_test_w, X_test_c
gc.collect()

# Meta-features (normalize)
scaler = StandardScaler()
meta_train_scaled = scaler.fit_transform(meta_train)
meta_test_scaled  = scaler.transform(meta_test)

# Combine
from scipy.sparse import csr_matrix
X_train_full = sp.hstack([X_train_tfidf, csr_matrix(meta_train_scaled)], format='csr')
X_test_full  = sp.hstack([X_test_tfidf,  csr_matrix(meta_test_scaled)],  format='csr')

y_train = train['Label'].values

print(f'Final feature shape: {X_train_full.shape}')
ram_usage()

In [ ]:
# Cross-validation
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

models = {
    'Logistic Regression': LogisticRegression(
        C=3, max_iter=500, solver='saga', class_weight='balanced'
    ),
    'LinearSVC': CalibratedClassifierCV(
        LinearSVC(C=0.5, max_iter=2000, class_weight='balanced'), cv=3
    )
}

print('Running Cross-Validation...\n')
results = {}

for name, model in models.items():
    scores = cross_val_score(
        model, X_train_full, y_train,
        cv=skf, scoring='f1_macro', n_jobs=1
    )
    results[name] = scores
    print(f'{name}:')
    print(f'  Folds: {[f"{s:.4f}" for s in scores]}')
    print(f'  Mean F1: {scores.mean():.4f} ± {scores.std():.4f}\n')

# Train best model
best_name = max(results, key=lambda k: results[k].mean())
best_ml   = models[best_name]
best_ml.fit(X_train_full, y_train)
baseline_probs = best_ml.predict_proba(X_test_full)
print(f'Best baseline: {best_name} | F1={results[best_name].mean():.4f}')

---
## 7. Feature Importance

Logistic Regression cho phép xem feature nào quan trọng nhất → debug & interpret model.

In [ ]:
# Chỉ chạy nếu best model là Logistic Regression
if 'Logistic Regression' in best_name:
    lr_model = best_ml
else:
    # Train LR riêng để visualize
    lr_model = LogisticRegression(C=3, max_iter=500, solver='saga', class_weight='balanced')
    lr_model.fit(X_train_full, y_train)

# Lấy feature names từ TF-IDF
word_features = vec_word.get_feature_names_out()
char_features = [f'char_{f}' for f in vec_char.get_feature_names_out()]
meta_names = [
    'word_count','char_count','avg_word_len',
    'exclaim_count','question_count','ellipsis_count',
    'negation_count','caps_ratio','title_ratio',
    'dr_mentions','year_mentions','star_mentions'
]
all_features = list(word_features) + list(char_features) + meta_names

coefficients = lr_model.coef_[0]
feature_df = pd.DataFrame({
    'feature': all_features[:len(coefficients)],
    'coef': coefficients
}).sort_values('coef', ascending=False)

print('=== Top Positive Sentiment Features ===')
print(feature_df.head(15).to_string(index=False))
print('\n=== Top Negative Sentiment Features ===')
print(feature_df.tail(15).to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_pos = feature_df.head(15)
axes[0].barh(top_pos['feature'], top_pos['coef'], color='#66c2a5')
axes[0].set_title('Top Positive Features')
axes[0].invert_yaxis()

top_neg = feature_df.tail(15)
axes[1].barh(top_neg['feature'], top_neg['coef'], color='#fc8d62')
axes[1].set_title('Top Negative Features')
axes[1].invert_yaxis()

plt.tight_layout(); plt.show()

# Kiểm tra NOT_ features có được học không
not_features = feature_df[feature_df['feature'].str.startswith('NOT_')]
print(f'\nNOT_ features trong top model: {len(not_features)}')
if not not_features.empty:
    print(not_features.head(10).to_string(index=False))

---
## 8. Token Dropout Augmentation

Kỹ thuật: randomly drop tokens trong training để tránh overfit.

Áp dụng khi augment data cho BERT.

In [ ]:
def token_dropout(text: str, dropout_prob: float = 0.1) -> str:
    """
    Lab 3.1: Randomly drop tokens from text for training augmentation.
    
    Example:
        "this movie was absolutely fantastic" (p=0.2)
        → "this absolutely fantastic"  (dropped 'movie was')
    """
    tokens = text.split()
    kept = [t for t in tokens if np.random.random() > dropout_prob]
    return ' '.join(kept) if kept else (tokens[0] if tokens else '')


# Test
print('=== Token Dropout Demo ===')
sample = "This product is absolutely fantastic and I totally love it"
print(f'Original: {sample}')
for i in range(3):
    print(f'Dropout {i+1}: {token_dropout(sample, 0.2)}')

print('\n→ Dùng trong BERT training để augment data (thêm vào ReviewDataset)')

---
## 9. BERT Fine-tuning — P100/T4 Compatible

In [ ]:
# Xóa EDA columns trước khi train BERT
train.drop(columns=['clean_text'], inplace=True, errors='ignore')
test.drop(columns=['clean_text'],  inplace=True, errors='ignore')
gc.collect()

MODEL_NAME   = 'distilbert-base-uncased'
MAX_LEN      = 128
BATCH_SIZE   = 32
EPOCHS       = 3
LR           = 3e-5        # ← Tăng từ 2e-5 lên 3e-5 (dataset lớn)
WARMUP_RATIO = 0.02        # ← Giảm từ 0.1 xuống 0.02 (dataset lớn)
N_FOLDS      = 3

print(f'Config: {MODEL_NAME} | MAX_LEN={MAX_LEN} | BS={BATCH_SIZE} | LR={LR} | warmup={WARMUP_RATIO}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ Tokenizer loaded')

In [ ]:
class ReviewDataset(Dataset):
    """Dataset với tùy chọn Token Dropout augmentation (Lab 3.1)."""
    def __init__(self, texts, labels=None, augment=False, dropout_prob=0.1):
        self.texts    = list(texts)
        self.labels   = list(labels) if labels is not None else None
        self.augment  = augment
        self.dropout_prob = dropout_prob

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Token dropout augmentation (Lab 3.1) — chỉ trong training
        if self.augment and self.dropout_prob > 0:
            text = token_dropout(text, self.dropout_prob)

        enc = tokenizer(
            text,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print('✅ ReviewDataset ready (with Token Dropout support)')

In [ ]:
scaler_amp = GradScaler('cuda', enabled=USE_AMP)

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    n_batches  = len(loader)
    for i, batch in enumerate(loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with autocast(device_type='cuda', enabled=USE_AMP):
            loss = model(**batch).loss
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item()
        if (i + 1) % max(1, n_batches // 4) == 0:
            print(f'    [{i+1}/{n_batches}] loss={loss.item():.4f}')
    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_probs = [], []
    for batch in loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
        with autocast(device_type='cuda', enabled=USE_AMP):
            logits = model(**inputs).logits
        probs = torch.softmax(logits.float(), dim=-1)
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_probs)

print('✅ Training functions ready')

In [ ]:
# 3-Fold CV
texts_all  = train['Text'].values
labels_all = train['Label'].values
texts_test = test['Text'].values

oof_preds       = np.zeros(len(train), dtype=int)
oof_probs       = np.zeros((len(train), 2))
test_probs_list = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(texts_all, labels_all)):
    print(f'\n{"="*55}')
    print(f'FOLD {fold+1}/{N_FOLDS}  |  train={len(tr_idx)}  val={len(val_idx)}')
    print(f'{"="*55}')
    ram_usage()

    # Dataset với augmentation trong training (Lab 3.1 Token Dropout)
    tr_loader   = DataLoader(
        ReviewDataset(texts_all[tr_idx], labels_all[tr_idx],
                      augment=True, dropout_prob=0.05),  # 5% token dropout
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader  = DataLoader(
        ReviewDataset(texts_all[val_idx], labels_all[val_idx]),
        batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0, pin_memory=True
    )
    test_loader = DataLoader(
        ReviewDataset(texts_test),
        batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0, pin_memory=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(DEVICE)

    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = len(tr_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_f1, best_state = 0, None

    for epoch in range(EPOCHS):
        print(f'\n  Epoch {epoch+1}/{EPOCHS}')
        avg_loss = train_one_epoch(model, tr_loader, optimizer, scheduler)
        val_preds, _ = evaluate(model, val_loader)
        f1 = f1_score(labels_all[val_idx], val_preds, average='macro')
        print(f'  → loss={avg_loss:.4f}  val_F1={f1:.4f}')
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'  ✅ New best: {best_f1:.4f}')

    model.load_state_dict(best_state)
    model.to(DEVICE)
    oof_preds[val_idx], oof_probs[val_idx] = evaluate(model, val_loader)
    _, test_probs = evaluate(model, test_loader)
    test_probs_list.append(test_probs)

    print(f'\n  Fold {fold+1} Final F1: {f1_score(labels_all[val_idx], oof_preds[val_idx], average="macro"):.4f}')

    del model, optimizer, scheduler, best_state
    del tr_loader, val_loader, test_loader
    torch.cuda.empty_cache(); gc.collect()
    ram_usage()

oof_f1 = f1_score(labels_all, oof_preds, average='macro')
print(f'\n{"="*55}')
print(f'🎯 OOF F1-macro (BERT {N_FOLDS}-fold): {oof_f1:.4f}')
print(f'{"="*55}')

---
## 10. Evaluation & Error Analysis

In [ ]:
print('=== BERT OOF Evaluation ===')
print(classification_report(labels_all, oof_preds,
      target_names=['Negative (0)', 'Positive (1)']))

cm = confusion_matrix(labels_all, oof_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Neg','Pred Pos'],
            yticklabels=['True Neg','True Pos'], ax=ax)
ax.set_title(f'OOF Confusion Matrix  |  F1={oof_f1:.4f}')
plt.tight_layout(); plt.show()

In [ ]:
# Error Analysis — Lab 3.1 approach
error_df = pd.DataFrame({
    'Text':       train['Text'],
    'Label':      labels_all,
    'oof_pred':   oof_preds,
    'prob_pos':   oof_probs[:, 1],
    'confidence': oof_probs.max(axis=1),
})
error_df['is_wrong'] = error_df['Label'] != error_df['oof_pred']
errors = error_df[error_df['is_wrong']].sort_values('confidence', ascending=False)

print(f'Errors: {len(errors)} / {len(error_df)} ({len(errors)/len(error_df)*100:.1f}%)')
print(f'False Pos (Neg→Pos): {(errors["oof_pred"]==1).sum()}')
print(f'False Neg (Pos→Neg): {(errors["oof_pred"]==0).sum()}')

print('\n--- Top 5 Confident Errors ---')
for _, r in errors.head(5).iterrows():
    print(f'Text: {r["Text"][:100]}...')
    print(f'True={r["Label"]} | Pred={r["oof_pred"]} | Conf={r["confidence"]:.3f}\n')

del error_df; gc.collect()

---
## 11. Submission

In [ ]:
# Ensemble BERT folds
test_avg_probs   = np.stack(test_probs_list, axis=-1).mean(axis=-1)
bert_final_preds = test_avg_probs.argmax(axis=1)

# Ensemble BERT + Baseline
BERT_WEIGHT = 0.80
ML_WEIGHT   = 0.20
ensemble_probs = BERT_WEIGHT * test_avg_probs + ML_WEIGHT * baseline_probs

# Threshold tuning trên OOF
best_thr, best_thr_f1 = 0.5, 0.0
for thr in np.arange(0.3, 0.71, 0.01):
    preds = (oof_probs[:, 1] >= thr).astype(int)
    f1 = f1_score(labels_all, preds, average='macro')
    if f1 > best_thr_f1:
        best_thr_f1 = f1
        best_thr    = thr

print(f'Default threshold 0.5: F1={oof_f1:.4f}')
print(f'Optimal threshold {best_thr:.2f}: F1={best_thr_f1:.4f} (+{(best_thr_f1-oof_f1)*100:.2f}%)')

final_preds = (ensemble_probs[:, 1] >= best_thr).astype(int)

submission = pd.DataFrame({'ID': range(1, len(test)+1), 'Label': final_preds})
submission.to_csv('submission.csv', index=False)

print('\n=== Summary ===')
print(f'Baseline CV F1:  {results[best_name].mean():.4f}')
print(f'BERT OOF F1:     {oof_f1:.4f}')
print(f'Optimal thr:     {best_thr:.2f} → F1={best_thr_f1:.4f}')
display(submission.head(5))
print(submission['Label'].value_counts())

In [ ]:
# Sanity check
sub = pd.read_csv('submission.csv')
checks = [
    ('Số dòng đúng',  len(sub) == len(test)),
    ('Có cột ID',     'ID' in sub.columns),
    ('Có cột Label',  'Label' in sub.columns),
    ('Label chỉ 0/1', set(sub['Label'].unique()).issubset({0,1})),
    ('Không có NaN',  sub.isnull().sum().sum() == 0),
    ('ID từ 1 đến N', sub['ID'].min()==1 and sub['ID'].max()==len(test)),
]
all_ok = True
for name, result in checks:
    print(f'{"✅" if result else "❌"} {name}')
    all_ok = all_ok and result
print('\n🚀 Sẵn sàng submit!' if all_ok else '⚠️ Có lỗi!')